# Eigenmodes and population dynamics

[Course index](../README.md) · [Week 12](../seminars/12_eigenvalues_and_dynamics.md)

**Predict → compute → explain → change an assumption.** Run top to bottom in a fresh Python kernel. GitHub previews do not execute widgets. Install the repository requirements once; no downloads occur in this notebook. All investigations have a paper route in the linked seminar sheets. A plot supports exploration, not a proof.

AI may help with the investigation if permitted. Write a prediction first, then independently verify at least one claim. Individual exit questions are completed without AI.

In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in the repository root or in notebooks/.
root = Path.cwd() if (Path.cwd() / 'la_labs.py').exists() else Path.cwd().parent
if not (root / 'la_labs.py').exists():
    raise RuntimeError('Start Jupyter in the repository root or notebooks directory.')
if str(root) not in sys.path: sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import la_labs as la


## Predict first

For a=.1,b=.2, solve Pπ=π with π₁+π₂=1 before running. What should the second eigenvalue control? Predict the boundary cases a=b=0 and a=b=1.

## Worked route: find what stays fixed and what changes

The [original transformation notebook](../6_linear_transformations.ipynb) shows how a matrix acts once. Here we apply the same matrix repeatedly. We look for directions whose images remain on the same line: $Pv=\lambda v$ with $v\ne0$. These are eigenvectors, and the scalar $\lambda$ tells us how that component changes at each step.

For $P=\begin{pmatrix}0.9&0.2\\0.1&0.8\end{pmatrix}$, a stationary probability vector satisfies $P\pi=\pi$. The first equation reduces to $0.1\pi_1=0.2\pi_2$, so $\pi_1=2\pi_2$. With total mass 1, this gives $\pi=(2/3,1/3)^T$. Its eigenvalue is 1: it stays fixed.

What about a different starting distribution? Its difference from $\pi$ has coordinate sum zero, so it is a multiple of $(1,-1)^T$. Direct multiplication gives

$$P(1,-1)^T=(0.7,-0.7)^T=0.7(1,-1)^T.$$

This separates the distribution into a stationary part and a shrinking part. After t steps,

$$p_t=\pi+0.7^t(p_0-\pi).$$

The column-sum convention matters: the jth column tells us where mass currently in state j goes next. Nonnegative entries and unit column sums preserve probability vectors. The experiment below lets you change those transfer fractions while retaining this model.

In [ ]:
def explore(a=.1, b=.2, p=1.):
    fig, orbit, stationary, eigenvalue = la.dynamics_figure(a,b,p)
    print('P =\n',la.markov_matrix(a,b))
    print('unique stationary vector:',stationary,'other eigenvalue:',eigenvalue)
    print('First six distributions:\n',orbit[:6].round(4))
    display(fig); plt.close(fig)
sliders = la.interactive_plot(explore, {'a': (0,1,.05,.1), 'b': (0,1,.05,.2), 'p': (0,1,.1,1)})

## Explain the formula

For a+b>0, π=(b,a)/(a+b). Prove that a zero-sum vector is an eigenvector for 1−a−b. Derive the whole orbit, then state precisely when it converges. A single plot cannot settle every parameter case.

### Follow the derivation into the exceptional cases

For $P(a,b)$ the stationarity equation is $a\pi_1=b\pi_2$. When $s=a+b>0$, normalization gives $\pi=(b/s,a/s)^T$. A zero-sum difference is multiplied by $1-s$. Thus all starts converge to this stationary vector when $0<s<2$, because $|1-s|<1$.

We must not use the division formula when $s=0$. Then $a=b=0$ and P is the identity: every distribution stays where it started. At the other boundary, $s=2$ forces $a=b=1$, and P swaps the states. Its error eigenvalue is −1, so a nonstationary distribution alternates without shrinking.

Between those boundaries a negative eigenvalue means alternating **and shrinking** errors. This is why “negative eigenvalue” alone is not a conclusion about divergence. Compare the exact formulas with the finite tables below.

In [ ]:
for a,b in [(.1,.2),(.8,.7),(0.,0.),(1.,1.)]:
    orbit, stationary, lam = la.markov_orbit(a,b,p=1.,steps=8)
    assert np.allclose(orbit.sum(axis=1),1)
    assert np.all(orbit >= -1e-14)
    if stationary is not None:
        predicted = stationary + lam**np.arange(9)[:,None]*(orbit[0]-stationary)
        assert np.allclose(orbit,predicted)
    print('a,b =',a,b,'other eigenvalue =',lam,'last two states =',orbit[-2:])

## Repeated eigenvalue challenge

For J=[[1,1],[0,1]], predict Jᵗ before running. Find the eigenspace by hand. Does eigenvalue 1 imply bounded iterates for every matrix?

### Why the eigenvalue list is not the whole story

For $J=\begin{pmatrix}1&1\\0&1\end{pmatrix}$, solving $(J-I)v=0$ forces $v_2=0$. There is only one independent eigenvector direction even though the characteristic polynomial is $(\lambda-1)^2$.

Write $J=I+N$, where $N=\begin{pmatrix}0&1\\0&0\end{pmatrix}$ and $N^2=0$. In the product $(I+N)^t$, all terms containing two or more factors N vanish. What remains is $I+tN$, so $J^t=\begin{pmatrix}1&t\\0&1\end{pmatrix}$ for nonnegative integer t. The image of $e_2$ is $(t,1)^T$, which grows despite the only eigenvalue being 1.

A diagonalization needs a whole basis of eigenvectors, not merely enough roots counted with multiplicity. The following powers make the distinction visible.

In [ ]:
J = np.array([[1,1],[0,1]],dtype=int)
for t in [0,1,2,10]:
    power = np.linalg.matrix_power(J,t)
    assert np.array_equal(power,np.array([[1,t],[0,1]]))
    print('t =',t,'J^t =\n',power)

## Group artifact and individual transfer

Submit one prediction, a table or plot, and an argument covering all initial distributions for your chosen parameter pair. Include an exceptional case.

**Individual check:** for a=.2,b=.3, find the stationary vector and contraction factor without running this notebook. The four-point rubric is in the assessment guide.